# Structured Data

Requirements:
- Use preferrably Python `3.12.13` venvs.
    - Using older version might not stream events chunk by chunk, instead it might wait for the **entire** response before printing it.
- Add a `.env` file in the same folder as this notebook with a fields:
    - ANTHROPIC_BASE_URL = `<url_here>`
    - ANTHROPIC_AUTH_TOKEN = "`<jwt_token_here>`"

## ATENTION

- Prefer "claude-4-5-haiku" for this notebook.

## 0 Setup

### 0.0 Libraries
- anthropic
- dotenv

In [1]:
# %pip install anthropic python-dotenv

### 0.1 Making a request
- API
- Auth and AI Model

In [2]:
# Env vars, Auth and AI model

from dotenv import load_dotenv
from anthropic import Anthropic

# Env vars

load_dotenv()

# Auth and AI model

client = Anthropic()

# model = "bedrock/anthropic.claude-4-6-sonnet"
model = "bedrock/anthropic.claude-4-5-haiku"

### 0.2 Functions
- Helpers

In [3]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=None,
):
    # Required
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    
    # Optional
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    # KW-args
    message = client.messages.create(**params)
    return message.content[0].text

## 1 Stop sequence

In [4]:
user_prompt = "Generate a very short event bridge rule as json."

In [5]:
# Without stop sequence

messages = []

add_user_message(messages, user_prompt)

chat(messages)

'```json\n{\n  "Name": "my-rule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:my-function",\n      "Id": "1"\n    }\n  ]\n}\n```'

In [6]:
# With stop sequence

messages = []

add_user_message(
    messages,
    user_prompt,
)

add_assistant_message(
    messages,
    "```json", # <---- mocked prefill
)

text = chat(
    messages, 
    stop_sequences=["```"], # <---- stop sequence
)

text

'\n{\n  "Name": "MyEventRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [7]:
import json

json.loads(text.strip())

{'Name': 'MyEventRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

## Exercise!

- Use message prefilling and stop sequences only yo get three different commands in a single response.
- There shouldn't be any comments or explanations.
- Hint: message prefilling isn't limited to just characters like ```.

In [8]:
messages = []

prompt = """"
Generate three different sample AWS CLI commands. It should be very short.
"""

add_user_message(
    messages, 
    prompt,
)

# Prefill sequence
prefill_sequence = """
Here are all three commands in a single block without any comments
```bash"""

add_assistant_message(
    messages,
    prefill_sequence
)

# Stop sequence
stop_sequence = "```"

response = chat(
    messages, 
    stop_sequences=[stop_sequence],
)

response.strip()

'aws s3 ls\naws ec2 describe-instances\naws dynamodb list-tables'